# QLoRA Training + Inference (Config C & D)

**One-stop notebook**: fine-tune Llama-3.1-8B with QLoRA, then immediately run
inference for Config C (fine-tuned) and Config D (base model) — no file transfer needed.

**Runtime**: Colab Pro — **A100 GPU** (recommended) or L4.

**Before running:**
1. Upload `train_sft.json` (generated by `build_sft_data.py`) when prompted.
2. Upload 9 `*.txt` files from `experiments/evaluation/data/topics/` when prompted.

**Outputs** (download at the end):
- `baseline_ft/*.json` — Config C: QLoRA-Llama-3.1-8B
- `baseline_base/*.json` — Config D: Llama-3.1-8B vanilla base

**Execution order:** Run all cells top-to-bottom.

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes xformers
!pip install datasets

In [ ]:
!nvidia-smi

In [ ]:
import json
import os
import re
import shutil
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import torch
from google.colab import files

# ---- Upload SFT training data ----
if not Path("train_sft.json").exists():
    print("Upload train_sft.json:")
    uploaded = files.upload()
    print(f"Loaded: {list(uploaded.keys())}")
else:
    print("train_sft.json already present.")

with open("train_sft.json") as f:
    sft_data = json.load(f)
total_turns = sum(len([m for m in s["messages"] if m["role"] != "system"]) for s in sft_data)
print(f"SFT samples: {len(sft_data)}  |  total dialogue turns: {total_turns}")

# ---- Upload topic source texts ----
topics_dir = Path("topics")
if not topics_dir.exists() or len(list(topics_dir.glob("*.txt"))) < 9:
    topics_dir.mkdir(exist_ok=True)
    print("\nUpload 9 topic .txt files (select all at once):")
    uploaded = files.upload()
    for fname, content in uploaded.items():
        (topics_dir / fname).write_bytes(content)
    print(f"Uploaded {len(uploaded)} topic files.")
else:
    print(f"topics/ already has {len(list(topics_dir.glob('*.txt')))} files.")

In [ ]:
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

gpu_name = torch.cuda.get_device_name(0)
print(f"GPU: {gpu_name}")

if "A100" in gpu_name or "H100" in gpu_name:
    MODEL_NAME = "unsloth/Llama-3.1-8B-Instruct"
    BATCH_SIZE = 4
elif "L4" in gpu_name:
    MODEL_NAME = "unsloth/Llama-3.1-8B-Instruct"
    BATCH_SIZE = 2
else:  # T4 fallback
    MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct"
    BATCH_SIZE = 2

print(f"Using model: {MODEL_NAME}  batch_size: {BATCH_SIZE}")

# ---- Load base model + LoRA ----
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
)

# ---- Prepare dataset ----
dataset = load_dataset("json", data_files="train_sft.json", split="train")

def to_text(examples):
    texts = []
    for msgs in examples["messages"]:
        text = tokenizer.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(to_text, batched=True, remove_columns=dataset.column_names)

# ---- Train ----
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    packing=False,
    args=TrainingArguments(
        output_dir="./output",
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        warmup_steps=5,
        learning_rate=1e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        save_strategy="no",
        optim="adamw_8bit",
        seed=42,
    ),
)

print("Starting training...")
stats = trainer.train()
print(f"Training complete. Final loss: {stats.training_loss:.4f}")

In [ ]:
# ============================================================
# Inference utilities (shared by Config C and D)
# ============================================================

TOPICS = [
    {"topic_id": "tech_1_transformer_attention", "topic": "Transformer Attention Mechanism", "domain": "technology"},
    {"topic_id": "tech_2_qlora_finetuning",      "topic": "QLoRA Efficient Fine-tuning",      "domain": "technology"},
    {"topic_id": "tech_3_multi_agent",            "topic": "LLM Multi-Agent Architecture",     "domain": "technology"},
    {"topic_id": "hum_1_ai_bias_fairness",        "topic": "AI Bias and Fairness",             "domain": "humanities"},
    {"topic_id": "hum_2_ai_privacy",              "topic": "AI Privacy and Data Protection",   "domain": "humanities"},
    {"topic_id": "hum_3_ai_safety_alignment",     "topic": "AI Safety and Alignment",          "domain": "humanities"},
    {"topic_id": "med_1_ai_medical_imaging",      "topic": "AI-Assisted Medical Imaging",      "domain": "medicine"},
    {"topic_id": "med_2_clinical_trials",         "topic": "Clinical Trial Design",            "domain": "medicine"},
    {"topic_id": "med_3_ai_drug_discovery",       "topic": "AI in Drug Discovery",             "domain": "medicine"},
]

MAX_NEW_TOKENS = 2048
TEMPERATURE    = 0.7
TOP_P          = 0.9


def load_source(topic_id: str) -> str:
    return (Path("topics") / f"{topic_id}.txt").read_text(encoding="utf-8")


def parse_turns(text: str) -> list:
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = re.sub(r"^```\w*\n?", "", cleaned)
        cleaned = re.sub(r"\n?```$", "", cleaned).strip()

    # 1) Direct parse
    try:
        data = json.loads(cleaned)
        if isinstance(data, list):
            return data
    except json.JSONDecodeError:
        pass

    # 2) Extract first [...] block
    m = re.search(r"\[.*\]", cleaned, re.DOTALL)
    if m:
        try:
            return json.loads(m.group(0))
        except json.JSONDecodeError:
            pass

    # 3) Missing closing bracket — trim trailing comma, truncate at last valid }
    if cleaned.startswith("[") and not cleaned.rstrip().endswith("]"):
        trimmed = cleaned.rstrip().rstrip(",")
        last_brace = trimmed.rfind("}")
        if last_brace > 0:
            candidate = trimmed[: last_brace + 1] + "]"
            try:
                parsed = json.loads(candidate)
                if isinstance(parsed, list) and len(parsed) > 1:
                    print(f"  [parse_turns] repaired truncated JSON -> {len(parsed)} turns")
                    return parsed
            except json.JSONDecodeError:
                pass

    return [{"speaker": "host", "text": cleaned}]


def generate_transcript(model, tokenizer, topic: dict, source_text: str) -> dict:
    """Single-prompt generation. System prompt matches SFT training format."""
    messages = [
        {
            "role": "system",
            "content": (
                "You are a podcast script writer. Given source material, write a natural "
                "two-person podcast conversation between a Host and an Expert.\n\n"
                f"Topic: {topic['topic']}\n\n"
                f"Source material:\n{source_text[:4000]}"
            ),
        },
        {
            "role": "user",
            "content": (
                "Requirements:\n"
                "- Generate exactly 10 turns (5 host lines + 5 expert lines, "
                "strictly alternating, starting with host).\n"
                "- The Host asks engaging questions and guides the conversation.\n"
                "- The Expert provides insightful, accurate answers grounded in the source material.\n"
                "- Each turn: 1-3 sentences, natural and suitable for audio.\n\n"
                'Return ONLY a JSON array (no markdown fences). '
                'Each element: {"speaker": "host" or "expert", "text": "..."}'
            ),
        },
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs    = tokenizer(prompt_text, return_tensors="pt").to("cuda")
    input_ids = inputs["input_ids"]
    input_len = input_ids.shape[-1]

    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()

    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            do_sample=True,
        )

    latency  = time.time() - t0
    peak_mem = torch.cuda.max_memory_allocated() / (1024 ** 3)

    output_ids   = outputs[0][input_len:]
    output_len   = len(output_ids)
    response_txt = tokenizer.decode(output_ids, skip_special_tokens=True).strip()
    turns        = parse_turns(response_txt)

    return {
        "turns": turns,
        "metrics": {
            "latency_seconds":   round(latency, 2),
            "input_tokens":      int(input_len),
            "output_tokens":     int(output_len),
            "total_tokens":      int(input_len + output_len),
            "estimated_cost_usd": 0.0,
            "peak_gpu_memory_gb": round(peak_mem, 3),
        },
    }


def run_config(model, tokenizer, config_name: str, model_label: str, out_dir: str):
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    print(f"\n{'=' * 55}")
    print(f"Config {config_name} — {model_label}")
    print(f"{'=' * 55}")

    for i, topic in enumerate(TOPICS, 1):
        json_path = out_path / f"{topic['topic_id']}.json"
        if json_path.exists():
            print(f"  [{i}/9] SKIP {topic['topic_id']} (exists)")
            continue

        source_text = load_source(topic["topic_id"])
        print(f"  [{i}/9] {topic['topic_id']} ...", end=" ", flush=True)

        result = generate_transcript(model, tokenizer, topic, source_text)

        full_result = {
            "config":       config_name,
            "topic_id":     topic["topic_id"],
            "topic":        topic["topic"],
            "domain":       topic["domain"],
            "generated_at": datetime.now(timezone.utc).isoformat(),
            "model":        model_label,
            "turns":        result["turns"],
            "metrics":      result["metrics"],
        }

        json_path.write_text(json.dumps(full_result, indent=2, ensure_ascii=False))
        m = result["metrics"]
        print(
            f"done  turns={len(result['turns'])}  "
            f"latency={m['latency_seconds']:.1f}s  "
            f"tokens={m['total_tokens']}  "
            f"gpu_mem={m['peak_gpu_memory_gb']:.2f}GB"
        )

    print(f"\nSaved to {out_path}/")


print("Inference utilities ready.")

In [ ]:
# ============================================================
# Config C — QLoRA fine-tuned model (already in memory)
# ============================================================

# model/tokenizer are already loaded from training — switch to inference mode
FastLanguageModel.for_inference(model)

run_config(model, tokenizer, "C", f"QLoRA-{MODEL_NAME.split('/')[-1]}", "baseline_ft")

# Free GPU memory before loading base model
del model, tokenizer
torch.cuda.empty_cache()
print("GPU memory freed.")

In [ ]:
# ============================================================
# Config D — Base model (no adapter)
# ============================================================

print(f"Loading base model: {MODEL_NAME} ...")
model_d, tokenizer_d = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
)
FastLanguageModel.for_inference(model_d)
print("Base model loaded.\n")

run_config(model_d, tokenizer_d, "D", MODEL_NAME.split("/")[-1], "baseline_base")

del model_d, tokenizer_d
torch.cuda.empty_cache()
print("Done.")

In [ ]:
# ============================================================
# Download results
# ============================================================

archive_name = "experiment_local_results"
shutil.make_archive(archive_name, "zip", ".", "baseline_ft")

with zipfile.ZipFile(f"{archive_name}.zip", "a") as zf:
    for f in Path("baseline_base").glob("*.json"):
        zf.write(f)

files.download(f"{archive_name}.zip")
print(f"Download started: {archive_name}.zip")
print("Extract baseline_ft/ → experiments/evaluation/data/Llama_ft/")
print("Extract baseline_base/ → experiments/evaluation/data/Llama_base/")